# 03 · Reproduce the chronological forecasting benchmark

This notebook trains the canonical regressor and classifier from scratch using the shared pipeline. It does **not** reuse the old notebooks’ accuracy claims.

**Task:** predict the next half-hour mean using lagged observed history. Train before July 2023, choose loss/cutoff on July–December 2023, refit and freeze before scoring 2024. Test observations become lagged history for subsequent one-step predictions; models are not retrained during test. No operational publication-latency claim is made.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nem_analysis.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from nem_analysis import load_market, filter_market, aggregate_market, smooth_prices, price_episodes, score_forecasts, season_labels
market, metadata = load_market()
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['#087e72', '#df9562', '#8296b0', '#233c35']
print(f"{len(market):,} half-hour positions | {market.index.min()}–{market.index.max()} | AEST (UTC+10)")

44,911 half-hour positions | 2022-01-01 00:00:00–2024-07-24 15:00:00 | AEST (UTC+10)


## Rebuild the experiment
Two gradient-boosting losses are compared on validation MAE. Random-forest spike classification uses a fixed target of price ≥ $300/MWh, with the cutoff chosen by validation F1. This CPU experiment normally takes seconds to a few minutes depending on the machine.

In [2]:
from scripts.build_data import prepare_data, benchmark
cleaned, _ = prepare_data()
from threadpoolctl import threadpool_limits
with threadpool_limits(limits=2):
    predictions, result = benchmark(cleaned)
leaderboard = pd.DataFrame(result['models']).sort_values('mae').reset_index(drop=True)
display(leaderboard)

Training gradient boosting (absolute_error)...


Training gradient boosting (squared_error)...


Training price-spike classifier...


,name,mae,rmse,bias
0,Last interval,41.643,385.487,0.002
1,Gradient boosting,45.032,550.877,-28.725
2,Previous day,87.490,718.379,-0.245
3,Previous week,105.441,793.464,-1.515


## Validate the saved benchmark
The included artifact was generated with the pinned Python dependencies. Small floating-point differences across platforms are possible; material differences should be investigated rather than silently accepted.

In [3]:
published = pd.DataFrame(metadata['benchmark']['models']).set_index('name')
reproduced = leaderboard.set_index('name')
comparison = reproduced[['mae','rmse','bias']].join(published[['mae','rmse','bias']],rsuffix='_published')
display(comparison)
assert result['test']['count'] == metadata['benchmark']['test']['count']
assert predictions.loc[predictions.index < '2024-01-01','prediction'].isna().all()
assert result['train']['end'] < result['validation']['start'] < result['test']['start']
for name in reproduced.index:
    np.testing.assert_allclose(reproduced.loc[name,['mae','rmse']].to_numpy(dtype=float), published.loc[name,['mae','rmse']].to_numpy(dtype=float), rtol=.001, atol=.01)
print('Chronology and published results reproduced within the documented tolerance.')

,mae,rmse,bias,mae_published,rmse_published,bias_published
name,,,,,,
Last interval,41.643,385.487,0.002,41.643,385.487,0.002
Gradient boosting,45.032,550.877,-28.725,45.032,550.877,-28.725
Previous day,87.490,718.379,-0.245,87.490,718.379,-0.245
Previous week,105.441,793.464,-1.515,105.441,793.464,-1.515


Chronology and published results reproduced within the documented tolerance.


## Compare actual prices with forecasts
The chart uses daily means for legibility. MAE and RMSE above use original paired half-hours.

In [4]:
tested = cleaned.join(predictions).dropna(subset=['prediction'])
daily, _ = aggregate_market(tested,'Daily')
fig = go.Figure()
for column,label,color in [('price','Observed','#233c35'),('prediction','Gradient boosting','#087e72'),('persistence','Last interval','#df9562')]:
    fig.add_trace(go.Scatter(x=daily.index,y=daily[column],name=label,line_color=color))
fig.update_layout(template='plotly_white',height=420,yaxis_title='Daily mean AUD/MWh',xaxis_title='Test interval ending · AEST',hovermode='x unified')
fig.show(renderer='plotly_mimetype')

## Rare-event classification and feature importance
Accuracy alone is misleading for a rare class. Report precision, recall, F1, average precision and prevalence. Importance measures model dependence on validation, not causality.

In [5]:
classifier = result['classifier']
display(pd.Series({k: classifier[k] for k in ['precision','recall','f1','averagePrecision','prevalence','probabilityCutoff']}).to_frame('full_test_value'))
display(pd.DataFrame(classifier['confusion'],index=['Actual negative','Actual positive'],columns=['Predicted negative','Predicted positive']))
display(pd.DataFrame(result['importance']).head(10))

,full_test_value
precision,0.555195
recall,0.740260
f1,0.634508
averagePrecision,0.701822
prevalence,0.023402
probabilityCutoff,0.541600


,Predicted negative,Predicted positive
Actual negative,9503,137
Actual positive,60,171


,feature,importance
0,price_lag_1,38.874
1,hour_sin,3.513
2,solar_lag_1,2.476
3,price_lag_2,1.996
4,price_lag_48,1.843
5,hour_cos,1.774
6,price_lag_336,1.711
7,temperature_lag_1,1.663
8,demand_lag_1,1.651
9,price_mean_48,1.563


## Interpretation and neural-network scope

The last-interval baseline wins on the full test period. Gradient boosting is not promoted as a proven improvement. Large RMSE reflects rare, very large misses retained in the raw target.

The legacy RNN/LSTM experiments are preserved in `archive/notebooks/LSTM.ipynb`. They are historical reference, not reproduced evidence. This rebuilt benchmark deliberately does not require TensorFlow or a GPU. A new LSTM experiment should use the same causal features, correctly aligned target sequences, separate train-only scaling and a **new untouched holdout**; repeatedly optimizing on this observed 2024 test would invalidate the test boundary.

The optional error reference in the dashboards is a fixed validation-residual width, not a calibrated prediction interval. See [the complete model contract](../docs/METHODOLOGY.md).